Script to get all measurements of all control sources given a csv of control source detections, one per control source.
In this notebook, that csv is 'all_final_controls_truncated.csv'.
This notebook should be run after pulsarbypulsar_truncation

In [1]:
import numpy as np

import pandas as pd

from astropy.coordinates import SkyCoord

from vasttools.query import Query

In [2]:
import logging
logging.basicConfig(level=201)

In [3]:
controls = pd.read_csv('all_final_controls_truncated.csv')
psr_df = pd.read_csv('paper_df_VASTcoverage.csv')

In [5]:
blank = pd.read_csv('all_final_control_measurements.csv')
blank.drop(blank.index, inplace=True)

In [6]:
blank.drop(columns=blank.columns[0], inplace=True)

In [ ]:
all_final_control_meas = []
names = psr_df['JNAME']
#names = controls['psr_name'].unique()

for name in names:
    print(name)
    container = blank
    
    name_mask = controls['psr_name']==name
    
    detection_mask = True##controls['num_detections']>30 
    for j in np.arange(controls[(name_mask)&(detection_mask)].shape[0]):
        print("     " + str(j))
        control_source = controls[
            (name_mask)
            &(detection_mask)
        ].iloc[j]
    
        mycoord = SkyCoord(str(control_source['ra_deg_cont']) + " " + str(control_source['dec_deg_cont']), unit='deg')
        my_query = Query(
            coords=mycoord, 
            epochs='all-vast', 
            use_tiles=True, 
            corrected_data=False
        )
        my_query.find_sources()
    
        control_meas = my_query.results[0].measurements
        control_meas.insert(0, 'PSR_assoc', name)
        control_meas.insert(1, 'control_number', j)
        all_final_control_meas.append(control_meas)
        container = pd.concat([container, control_meas])
    container.to_csv('controls_w_meas/' + name + '.csv')

J1702-4128
     0
     1
     2
     3
     4
     5
     6
     7
     8
     9
     10
     11
     12
     13
     14
     15
     16
     17
     18
     19
     20
     21
     22
     23
     24
     25
     26
     27
     28
     29
     30
     31
     32
     33
     34
     35
     36
     37
     38
     39
     40
     41
     42
     43
     44
     45
     46
J1628-4804
     0
     1
     2
     3
     4
     5
     6
     7
     8
     9
     10
     11
     12
     13
     14
     15
     16
     17
     18
     19
     20
     21
     22
     23
     24
     25
     26
     27
     28
     29
     30
     31
     32
     33
     34
     35
     36
     37
     38
     39
     40
     41
     42
     43
     44
     45
     46
     47
     48
     49
     50
     51
     52
     53
     54
     55
     56
     57
     58
     59
     60
     61
     62
     63
     64
     65
     66
     67
J1840-1207
     0
     1
     2
     3
     4
     5
     6
     7
     8
    

In [4]:
all_final_control_meas = []
names = psr_df['JNAME']

for name in names:
    control_meas = pd.read_csv('controls_w_meas/' + name + '.csv')
    all_final_control_meas.append(control_meas)

In [12]:
for i in np.arange(len(all_final_control_meas)):
    df = all_final_control_meas[i]
    df.drop(columns=df.columns[0], inplace=True)

In [14]:
pd.concat(all_final_control_meas).to_csv('all_final_control_measurements.csv')